# ⚙️ Der Optimizer ist dein Compiler

Traditionell: **Quellcode → Compiler → Binary**
Heute: **Signature + Metrik + Daten → Der Optimizer → Optimierter Prompt**

Beides nimmt menschenlesbare Spezifikationen und produziert maschinenausführbare Artefakte.

Im letzten Notebook hast du gesehen: selbst wenn du dem Modell die gültigen Kategorien, Prioritäten und Gruppennamen gibst, bleibt der Score bei ~30%. Kann ein **automatischer Optimizer** das besser?

## Was passiert hier?

Im letzten Notebook hast du Prompts **manuell** verbessert. Jetzt lässt du den **Computer** das machen.

Das Prinzip ist einfach:
1. Du sagst, **was** du willst (Signature)
2. Du sagst, **was gut heisst** (Metrik)
3. Du gibst **Beispiele** (Daten)
4. Der Optimizer findet automatisch den besten Prompt

Das ist wie ein Compiler: Du schreibst Quellcode, der Compiler macht eine optimierte Binary. Hier schreibst du eine Spezifikation, der Optimizer macht einen optimierten Prompt.


In [1]:
import sys
sys.path.insert(0, ".")
from dspy_tasks.config import get_available_models, configure_dspy

# Verfügbare Modelle
print("Verfügbare Modelle:")
for m in get_available_models()[:10]:
    print(f"  • {m}")

# Modell wählen (ändere den String um ein anderes zu nutzen)
MODEL = "github_copilot/gpt-5.1"
configure_dspy(MODEL)
print(f"\n✅ Konfiguriert: {MODEL}")

Verfügbare Modelle:
  • github_copilot/gpt-4o
  • github_copilot/claude-sonnet-4
  • github_copilot/gpt-4o-mini
  • github_copilot/gpt-3.5-turbo-0613
  • github_copilot/claude-sonnet-4.5
  • github_copilot/gemini-3-pro-preview
  • github_copilot/gpt-4
  • github_copilot/gpt-5.3-codex
  • github_copilot/gpt-4.1
  • github_copilot/gpt-4o-2024-11-20

✅ Konfiguriert: github_copilot/gpt-5.1


### 🔄 Der Optimizer-Workflow als Diagramm

So sieht automatische Optimierung aus — in vier Schritten. Du lieferst die **Zutaten** (Signature, Metrik, Daten), der Optimizer **kocht** daraus den besten Prompt.

Das Schöne: Wenn sich deine Daten ändern, optimierst du einfach nochmal. Der Prozess ist reproduzierbar.


In [2]:
from dspy_tasks.visualize import diagram

diagram([
    {"label": "Signature", "detail": "Was du willst", "icon": "📝", "color": "#0078d4"},
    {"label": "Metrik", "detail": "Was 'gut' heisst", "icon": "📐", "color": "#0078d4"},
    {"label": "Trainingsdaten", "detail": "Beispiele", "icon": "📊", "color": "#0078d4"},
    {"label": "Optimizer", "detail": "probiert Varianten", "icon": "⚙️", "color": "#ca5010"},
    {"label": "Optimierter Prompt", "detail": "+ Few-Shot Demos", "icon": "🎯", "color": "#107c10"},
], title="Die Optimierungs-Pipeline")

## ✏️ Erst du, dann die Maschine

Bevor wir den automatischen Optimizer loslassen, versuch es nochmal selbst! Das ist dieselbe Ticket-Routing-Aufgabe aus Notebook 01. 

Editiere den Prompt unten und schau, welchen Score du erreichst. Merk dir deinen besten Score — danach vergleichen wir mit dem automatisch optimierten Ergebnis.

### 🤔 Warum erst manuell?

Gute Frage! Wir wollen, dass du ein **Gefühl** für Prompt-Engineering bekommst. Wenn du selbst versucht hast, einen Prompt zu verbessern, verstehst du viel besser, was der Optimizer tut — und warum er es besser kann als wir.

Merk dir deinen Score — gleich vergleichen wir ihn mit dem des Optimizers.


In [ ]:
from dspy_tasks.actions import run_with_prompt
from dspy_tasks.visualize import display_score, display_results_table

# Dein manueller Versuch — ändere den Prompt und führe die Zelle erneut aus!
MEIN_PROMPT = """Classify this IT support ticket.
Category MUST be one of: Event NO Customer Impact, Failure, Service Request
Priority MUST be one of: High, Medium, Standard
Assigned group MUST be one of: SDE - Service Desk, OFC - Office & Collaboration, PRM - Premium Support, CDC - Client Design & Standard SW Integration, OUM - Incident, Helpdesk, Service-Center IKT, Service-Center IKT Bestellungen, Smartcard Office, CBCD - Container Basierte Cloud Dienste, OPC - Applikationen, DevOps - Rein, BVX - ePortal Service Line, IOM - Input / Output Mgmt., OPM - DLC Dispatching, O-SDK, ESTV-RSS-Stammdaten, FIB - BIT Store Bollwerk, Immobilien BAZG, Bedarfsmanagement, Güter und Ausrüstung"""

result = run_with_prompt("ticket_routing", MEIN_PROMPT, max_eval=3)
display_score("Dein manueller Prompt", result.score)
display_results_table(result.individual_scores)

  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] ✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓


## ⚙️ Und jetzt die Maschine...

Du hast deinen besten manuellen Score gesehen. Selbst MIT allen gültigen Werten im Prompt liegt er vermutlich bei 30-50%. Das Modell kennt die Werte, aber es weiss nicht, **wann welcher Wert passt**.

Jetzt drück unten auf "Optimieren" und schau, was passiert. Der Optimizer lernt aus den Trainingsdaten die **Zuordnungsregeln** und findet automatisch den besten Prompt.

**Die Frage ist:** Kann der Computer einen besseren Prompt finden als du?

## BootstrapFewShot: Der schnelle Compiler

**BootstrapFewShot** ist wie `-O1` Optimierung — schnell und effektiv. Er sucht die besten Few-Shot-Beispiele aus deinen Trainingsdaten und fügt sie in den Prompt ein.

Beim Ticket-Routing heisst das: Der Optimizer wählt automatisch die informativsten Beispiel-Tickets aus, damit das Modell lernt: *"Account gesperrt" → SDE - Service Desk, "CPAM Ausfall" → CDC*.

Dauer: ~10 Sekunden. Verbesserung: oft 10-30%.

### 📋 Was macht BootstrapFewShot genau?

Stell dir vor, du hast 35 Trainings-Tickets. BootstrapFewShot probiert verschiedene Kombinationen durch und findet heraus: *Welche 3-5 Beispiel-Tickets im Prompt liefern die besten Ergebnisse?*

Es ist wie ein Koch, der verschiedene Gewürzkombinationen testet — systematisch statt nach Bauchgefühl.

In [4]:
from dspy_tasks.actions import run_optimization
from dspy_tasks.tasks import get_task
from dspy_tasks.visualize import display_improvement, display_insight, display_prompt_diff

task = get_task("ticket_routing")
print(f"⏳ Optimiere {task.name} mit BootstrapFewShot...")
print(f"   Das kann 10-60 Sekunden dauern...\n")

result = run_optimization("ticket_routing", "BootstrapFewShot", max_eval=8)

display_improvement(result.baseline_score, result.optimized_score)
print(f"⏱️  Optimierung dauerte {result.elapsed_seconds}s | {result.llm_calls} LLM-Aufrufe")

display_prompt_diff(result.prompt_before, result.prompt_after)

display_insight("Was gerade passiert ist",
    f"Der Optimizer hat {result.optimized_score:.0%} erreicht vs. Baseline {result.baseline_score:.0%}. "
    "Er hat aus den Trainingsdaten die besten Beispiel-Tickets ausgewählt und in den Prompt eingefügt.")

⏳ Optimiere Ticket Classification & Routing mit BootstrapFewShot...
   Das kann 10-60 Sekunden dauern...

  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] ✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓


 14%|█▍        | 5/35 [00:21<02:09,  4.32s/it]

Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
  [1/8] 

✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] ✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓


⏱️  Optimierung dauerte 21.61s | 16 LLM-Aufrufe


## MIPROv2: Der Heavy-Duty Compiler

**MIPROv2** ist wie `-O3` — er optimiert gleichzeitig die **Instruktionen UND die Beispiele** mit Bayesian Search. Mächtiger, aber langsamer.

Dauer: ~30-60 Sekunden. Verbesserung: oft nochmal besser als BootstrapFewShot.

### 🔀 BootstrapFewShot vs. MIPROv2 — der direkte Vergleich

Zwei verschiedene Strategien treten gegeneinander an:

| Optimizer | Was er optimiert | Stärke |
|---|---|---|
| **BootstrapFewShot** | Welche Beispiele im Prompt stehen | Schnell, zuverlässig |
| **MIPROv2** | Beispiele UND Anweisungen | Gründlicher, aber langsamer |

Welcher gewinnt? Klick den Button und finde es heraus. Spoiler: Es hängt vom Task ab!


In [ ]:
from dspy_tasks.visualize import bar_comparison

task = get_task("ticket_routing")
print(f"⏳ Vergleiche Optimizer auf {task.name}...\n")

r_bs = run_optimization("ticket_routing", "BootstrapFewShot", max_eval=8)
print(f"BootstrapFewShot: {r_bs.baseline_score:.0%} → {r_bs.optimized_score:.0%} ({r_bs.elapsed_seconds}s)")

r_mipro = run_optimization("ticket_routing", "MIPROv2", max_eval=8)
print(f"MIPROv2:          {r_mipro.baseline_score:.0%} → {r_mipro.optimized_score:.0%} ({r_mipro.elapsed_seconds}s)")

scores = {
    "BootstrapFewShot": {"baseline": r_bs.baseline_score, "optimized": r_bs.optimized_score},
    "MIPROv2": {"baseline": r_mipro.baseline_score, "optimized": r_mipro.optimized_score},
}
fig = bar_comparison("Ticket Routing: Optimizer-Vergleich", scores)
fig.show()


⏳ Vergleiche Optimizer auf Ticket Classification & Routing...

  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] ✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓


 14%|█▍        | 5/35 [00:00<00:00, 47.34it/s]

Bootstrapped 4 full traces after 5 examples for up to 1 rounds, amounting to 5 attempts.
  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] 

✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓
BootstrapFewShot: 5% → 54% (0.11s)
  [1/8] ✓
  [2/8] ✓
  [3/8] ✓
  [4/8] ✓
  [5/8] ✓
  [6/8] ✓
  [7/8] ✓
  [8/8] ✓


2026/03/24 17:14:57 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 10
minibatch: False
num_fewshot_candidates: 6
num_instruct_candidates: 3
valset size: 28

2026/03/24 17:14:57 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2026/03/24 17:14:57 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2026/03/24 17:14:57 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=6 sets of demonstrations...


...


Average Metric: 2.00 / 28 (7.1%): 100%|██████████| 28/28 [00:11<00:00,  2.51it/s]

2026/03/24 17:17:48 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 28 (7.1%)
2026/03/24 17:17:48 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 7.14 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 0'].
2026/03/24 17:17:48 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [1.43, 10.71, 7.14]
2026/03/24 17:17:48 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 10.71
2026/03/24 17:17:48 INFO dspy.teleprompt.mipro_optimizer_v2: ========================


2026/03/24 17:17:48 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 4 / 10 =====



Average Metric: 3.00 / 28 (10.7%): 100%|██████████| 28/28 [00:11<00:00,  2.46it/s]

2026/03/24 17:17:59 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 28 (10.7%)
2026/03/24 17:17:59 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 10.71 with parameters ['Predictor 0: Instruction 1', 'Predictor 0: Few-Shot Set 5'].
2026/03/24 17:17:59 INFO dspy.teleprompt.mipro_optimizer_v2: Scores so far: [1.43, 10.71, 7.14, 10.71]
2026/03/24 17:17:59 INFO dspy.teleprompt.mipro_optimizer_v2: Best score so far: 10.71
2026/03/24 17:17:59 INFO dspy.teleprompt.mipro_optimizer_v2: ========================


2026/03/24 17:17:59 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 5 / 10 =====



Average Metric: 3.00 / 27 (11.1%):  93%|█████████▎| 26/28 [00:12<00:01,  1.84it/s]

### 🧪 Beliebigen Task optimieren

Jetzt kannst du selbst experimentieren: Ändere den TASK-String oben und sieh dir den Vorher-Nachher-Unterschied an.

Der Optimizer zeigt dir genau, **was** er am Prompt geändert hat. So lernst du, welche Formulierungen gut funktionieren — und kannst das Wissen auf eigene Prompts übertragen.


## 🎯 Beliebige Aufgabe optimieren

Wähl eine Aufgabe und einen Optimizer — und schau dir an, was sich verändert. Der Prompt-Diff zeigt dir genau, was der Optimizer anders macht als du.


In [ ]:
# Weitere Aufgaben zum Optimieren:
available = ["ticket_routing", "multihop_qa", "report_generation"]
for tid in available:
    t = get_task(tid)
    print(f"  • {tid}: {t.name}")

# Wähle eine Aufgabe (ändere den String):
TASK = "ticket_routing"

print(f"\n⏳ Optimiere {TASK}...")
result = run_optimization(TASK, "BootstrapFewShot", max_eval=8)
display_improvement(result.baseline_score, result.optimized_score)
display_prompt_diff(result.prompt_before, result.prompt_after)

## 🏆 Vergleich: Du vs. Maschine

Schau dir den Unterschied an:
- **Dein bester manueller Prompt:** Selbst mit allen gültigen Werten wahrscheinlich ~30-50%
- **Automatisch optimierter Prompt:** Der Optimizer nutzt Few-Shot-Beispiele und findet bessere Formulierungen

Der optimierte Prompt enthält oft:
- **Automatisch ausgewählte Beispiel-Tickets** — die informativsten aus den Trainingsdaten
- **Präzisere Anweisungen** — Formulierungen die du vielleicht nicht probiert hättest
- **Implizites Wissen** — das Modell lernt Zuordnungsregeln aus den Beispielen

> 💡 **Das ist der Punkt:** Manuelles Prompt-Tuning funktioniert, ist aber langsam und fragil. Automatische Optimierung findet bessere Prompts, schneller, reproduzierbar. **Deine Metrik + Daten = dein Programm. Der Optimizer ist der Compiler.**

## ⏭️ Weiter geht's!

Der Optimizer hat den Prompt verbessert — automatisch, messbar, reproduzierbar. Aber was passiert, wenn du **deine eigenen, echten Daten** benutzt?

Im nächsten Notebook nehmen wir die echten Ticket-Daten aus dem Projekt und zeigen: **Deine Daten sind dein Burggraben.** Ein generisches Modell + deine Domain-Daten + Tuning = etwas, das kein Konkurrent kopieren kann.

👉 **[Weiter zu Notebook: Domain-Tuning →](03_domain_tuning.ipynb)**
